## Simple analysis of the TMDB clean data

In [21]:
# read the data
import os
import ast
import pandas as pd

# go up from the notebook folder until we find the project's "data" folder
ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "data")) and ROOT != os.path.dirname(ROOT):
    ROOT = os.path.dirname(ROOT)

tmdb_df = pd.read_csv(os.path.join(ROOT, "data/processed/tmdb_clean.csv"))

In [22]:
tmdb_df.columns

Index(['tmdb_id', 'title', 'original_title', 'release_date', 'release_year',
       'genre_ids', 'popularity', 'vote_average', 'vote_count',
       'original_language'],
      dtype='str')

In [23]:
# Dataset dimensions
print("Shape:", tmdb_df.shape)

# Column types and non-null counts
tmdb_df.info()

# First rows
display(tmdb_df.head())

# Descriptive statistics for numerical columns
display(tmdb_df.describe())

Shape: (76714, 10)
<class 'pandas.DataFrame'>
RangeIndex: 76714 entries, 0 to 76713
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tmdb_id            76714 non-null  int64  
 1   title              76714 non-null  str    
 2   original_title     76714 non-null  str    
 3   release_date       76714 non-null  str    
 4   release_year       76714 non-null  int64  
 5   genre_ids          76714 non-null  str    
 6   popularity         76714 non-null  float64
 7   vote_average       76714 non-null  float64
 8   vote_count         76714 non-null  int64  
 9   original_language  76714 non-null  str    
dtypes: float64(2), int64(3), str(5)
memory usage: 5.9 MB


,tmdb_id,title,original_title,release_date,release_year,genre_ids,popularity,vote_average,vote_count,original_language
0,299534,Avengers: Endgame,Avengers: Endgame,2019-04-24,2019,"[12, 878, 28]",98.7212,8.200,28768,en
1,496243,Parasite,기생충,2019-05-30,2019,"[35, 53, 18]",35.6564,8.490,21362,ko
2,466272,Once Upon a Time... in Hollywood,Once Upon a Time... in Hollywood,2019-07-24,2019,"[35, 18, 53]",34.9897,7.419,15284,en
3,475557,Joker,Joker,2019-10-01,2019,"[80, 53, 18]",34.4579,8.100,28367,en
4,429617,Spider-Man: Far From Home,Spider-Man: Far From Home,2019-06-28,2019,"[28, 12, 878]",33.8307,7.392,17646,en


,tmdb_id,release_year,popularity,vote_average,vote_count
count,7.671400e+04,76714.000000,76714.000000,76714.000000,76714.000000
mean,1.129443e+06,2022.489819,2.472966,3.144676,59.420275
std,3.576114e+05,2.285612,5.408205,3.407365,478.612582
min,5.492000e+03,2019.000000,1.103700,0.000000,0.000000
25%,8.206742e+05,2020.000000,1.627625,0.000000,0.000000
50%,1.117479e+06,2022.000000,1.874150,0.500000,1.000000
75%,1.436463e+06,2024.000000,2.288700,6.300000,6.000000
max,1.774995e+06,2026.000000,633.508500,10.000000,28768.000000


In [24]:
# popularity and vote_count are very skewed (a few blockbusters, many small movies)
# -> compare the mean with the median and the high percentiles
display(tmdb_df[["popularity", "vote_count"]].quantile([0.25, 0.5, 0.75, 0.9, 0.99, 1.0]))

print("popularity -> mean:", round(tmdb_df["popularity"].mean(), 2),
      "| median:", round(tmdb_df["popularity"].median(), 2))
print("vote_count -> mean:", round(tmdb_df["vote_count"].mean(), 2),
      "| median:", round(tmdb_df["vote_count"].median(), 2))

,popularity,vote_count
0.25,1.627625,0.00
0.50,1.874150,1.00
0.75,2.288700,6.00
0.90,3.081670,48.00
0.99,12.784025,1231.87
1.00,633.508500,28768.00


popularity -> mean: 2.47 | median: 1.87
vote_count -> mean: 59.42 | median: 1.0


In [25]:
# Missing values
print("Missing values per column:")
display(tmdb_df.isnull().sum())

# Duplicate rows
print("Duplicate rows:", tmdb_df.duplicated().sum())

# Duplicate TMDB ids (must be 0: one id = one movie)
print("Duplicate tmdb_id:", tmdb_df["tmdb_id"].duplicated().sum())

# Duplicate title + release date (must be 0 after the cleaning)
print("Duplicate title + release_date:", tmdb_df.duplicated(subset=["title", "release_date"]).sum())

# Duplicate titles alone can be normal: different movies can share the same title
print("Duplicate titles (homonyms, can be normal):", tmdb_df["title"].duplicated().sum())

# Number of unique movies and years
print("Unique movies (tmdb_id):", tmdb_df["tmdb_id"].nunique())
print("Years covered:", sorted(int(y) for y in tmdb_df["release_year"].dropna().unique()))

Missing values per column:


tmdb_id              0
title                0
original_title       0
release_date         0
release_year         0
genre_ids            0
popularity           0
vote_average         0
vote_count           0
original_language    0
dtype: int64

Duplicate rows: 0
Duplicate tmdb_id: 0
Duplicate title + release_date: 0
Duplicate titles (homonyms, can be normal): 6915
Unique movies (tmdb_id): 76714
Years covered: [2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [26]:
# Preparation (original columns are kept, we only add new ones)
tmdb_df["release_date"] = pd.to_datetime(tmdb_df["release_date"], errors="coerce")

# movies that nobody voted for
tmdb_df["no_votes"] = tmdb_df["vote_count"] == 0

# vote_average = 0 with 0 votes means "nobody voted", not a real 0/10 rating
# -> we keep the average only for movies that have at least one vote
tmdb_df["vote_average_voted"] = tmdb_df["vote_average"].where(tmdb_df["vote_count"] > 0)

print("Movies with no vote:", tmdb_df["no_votes"].sum(),
      f"({tmdb_df['no_votes'].mean() * 100:.1f}%)")

Movies with no vote: 38305 (49.9%)


In [27]:
cols = ["title", "release_year", "popularity", "vote_average", "vote_count", "original_language"]

# Most popular movies
display(tmdb_df.nlargest(10, "popularity")[cols])

# Most voted movies
display(tmdb_df.nlargest(10, "vote_count")[cols])

# Best rated movies (at least 1000 votes, otherwise the average is not reliable)
display(tmdb_df[tmdb_df["vote_count"] >= 1000].nlargest(10, "vote_average")[cols])

,title,release_year,popularity,vote_average,vote_count,original_language
67337,Spider-Man: Brand New Day,2026,633.5085,7.861,2871,en
67338,Resident Evil,2026,513.2853,7.300,455,en
67339,The End of Oak Street,2026,413.3113,7.004,1498,en
67340,Coyote vs. Acme,2026,347.0415,7.555,515,en
67341,The Odyssey,2026,333.2210,8.018,3899,en
67342,The Love Hypothesis,2026,273.1621,8.340,156,en
67343,The Mongoose,2026,249.4488,0.000,0,en
67344,Colony,2026,243.0215,8.071,876,ko
67345,Mutiny,2026,209.3601,6.736,497,en
67346,Zip Wire,2026,173.7655,7.600,14,en


,title,release_year,popularity,vote_average,vote_count,original_language
0,Avengers: Endgame,2019,98.7212,8.200,28768,en
3,Joker,2019,34.4579,8.100,28367,en
19217,Spider-Man: No Way Home,2021,60.5141,7.947,23182,en
1,Parasite,2019,35.6564,8.490,21362,ko
4,Spider-Man: Far From Home,2019,33.8307,7.392,17646,en
11,Captain Marvel,2019,25.2315,6.770,17173,en
19219,Dune,2021,38.4923,7.784,15586,en
2,Once Upon a Time... in Hollywood,2019,34.9897,7.419,15284,en
28824,Avatar: The Way of Water,2022,43.1055,7.583,14671,en
7,Knives Out,2019,29.5878,7.844,14526,en


,title,release_year,popularity,vote_average,vote_count,original_language
67373,Avatar Aang: The Last Airbender,2026,75.6223,9.147,1187,en
67414,Swapped,2026,43.0170,8.870,2203,en
57725,Demon Slayer: Kimetsu no Yaiba Infinity Castle,2025,57.1426,8.800,2173,ja
67361,Project Hail Mary,2026,99.5410,8.640,7959,en
67394,Michael,2026,53.8699,8.629,4407,en
57746,Chainsaw Man - The Movie: Reze Arc,2025,32.1940,8.600,1200,ja
1,Parasite,2019,35.6564,8.490,21362,ko
9908,Gabriel's Inferno,2020,4.5903,8.380,2482,en
38467,Spider-Man: Across the Spider-Verse,2023,54.5130,8.350,9291,en
67347,Toy Story 5,2026,170.4058,8.342,2295,en


In [28]:
# Number of movies and average popularity / rating per year
yearly_analysis = tmdb_df.groupby("release_year").agg(
    number_of_movies=("tmdb_id", "count"),
    mean_popularity=("popularity", "mean"),
    median_popularity=("popularity", "median"),
    median_vote_count=("vote_count", "median"),
    mean_vote_average=("vote_average_voted", "mean"),
    median_vote_average=("vote_average_voted", "median"),
    no_votes_pct=("no_votes", lambda s: s.mean() * 100)
).round(2)

display(yearly_analysis)

,number_of_movies,mean_popularity,median_popularity,median_vote_count,mean_vote_average,median_vote_average,no_votes_pct
release_year,,,,,,,
2019,9605,2.15,1.74,1.0,6.06,6.17,40.10
2020,9611,2.02,1.70,1.0,6.07,6.14,44.03
2021,9606,2.18,1.76,1.0,6.15,6.20,44.53
2022,9645,2.26,1.82,1.0,6.11,6.15,43.55
2023,9647,2.34,1.88,1.0,6.26,6.30,48.89
2024,9605,2.43,1.91,0.0,6.55,6.50,54.78
2025,9618,2.71,2.05,0.0,6.68,6.60,56.50
2026,9377,3.74,2.11,0.0,6.61,6.67,67.52


In [29]:
# Original languages (top 10)
language_counts = tmdb_df["original_language"].value_counts()

language_analysis = pd.DataFrame({
    "number_of_movies": language_counts,
    "share_%": (language_counts / len(tmdb_df) * 100).round(1)
}).head(10)

display(language_analysis)
print("Number of different languages:", tmdb_df["original_language"].nunique())

,number_of_movies,share_%
original_language,,
en,37578,49.0
es,6922,9.0
fr,5406,7.0
pt,3677,4.8
de,2527,3.3
ja,1945,2.5
it,1596,2.1
ko,1184,1.5
zh,1045,1.4


Number of different languages: 144


In [30]:
# Genres: genre_ids is stored as text like "[12, 878, 28]" -> we convert it to a real list
GENRES = {
    28: "Action", 12: "Adventure", 16: "Animation", 35: "Comedy", 80: "Crime",
    99: "Documentary", 18: "Drama", 10751: "Family", 14: "Fantasy", 36: "History",
    27: "Horror", 10402: "Music", 9648: "Mystery", 10749: "Romance",
    878: "Science Fiction", 10770: "TV Movie", 53: "Thriller", 10752: "War", 37: "Western",
}

# one row per (movie, genre)
genre_df = tmdb_df.assign(genre_id=tmdb_df["genre_ids"].apply(ast.literal_eval)).explode("genre_id")
genre_df["genre"] = genre_df["genre_id"].map(GENRES)

print("Rows without a known genre:", genre_df["genre"].isna().sum())

# a movie can have several genres, so the shares add up to more than 100%
genre_analysis = genre_df.dropna(subset=["genre"]).groupby("genre").agg(
    number_of_movies=("tmdb_id", "count"),
    median_popularity=("popularity", "median"),
    median_vote_average=("vote_average_voted", "median")
).sort_values("number_of_movies", ascending=False).round(2)

genre_analysis["share_%"] = (genre_analysis["number_of_movies"] / len(tmdb_df) * 100).round(1)

display(genre_analysis)

Rows without a known genre: 9199


,number_of_movies,median_popularity,median_vote_average,share_%
genre,,,,
Drama,28301,1.91,6.35,36.9
Comedy,14987,1.94,6.20,19.5
Documentary,12804,1.81,7.00,16.7
Thriller,9166,2.01,6.00,11.9
Horror,8666,1.92,5.75,11.3
Romance,6583,1.99,6.30,8.6
Action,4553,2.22,6.29,5.9
Animation,3946,1.96,6.90,5.1
Crime,3805,2.04,6.20,5.0


### Observations

- Le fichier contient 79 244 films (2019-2026), sans valeur manquante et sans doublon (`tmdb_id`, ni titre + date de sortie).
- Le nombre de films par année (environ 9 600 à 10 000) vient de la limite de l'API (10 000 films par an), pas du nombre réel de sorties.
- Les distributions sont très asymétriques : popularité médiane de 1,87 contre un maximum de 633,5. Environ la moitié des films n'ont aucun vote (`vote_count` médian = 0).
- La part de films sans vote passe de 41 % (2019) à 68 % (2026).
- Les 10 films les plus populaires sont tous de 2026, alors que 7 des 10 films les plus votés sont de 2019 : la popularité est un instantané, les votes s'accumulent avec le temps.
- Note moyenne (films ayant au moins un vote) : 6,06 en 2019, 6,68 en 2025. À interpréter avec prudence : pour les années récentes, seuls les films ayant reçu des votes sont comptés.
- Langues : anglais 49,2 %, espagnol 9,0 %, français 7,0 %.
- Genres : Drame 37,0 %, Comédie 19,3 %, Documentaire 16,5 %. Note médiane la plus haute : Documentaire et Musique (7,00), puis Animation (6,90). La plus basse : Horreur (5,77). La popularité médiane varie peu selon les genres (1,81 à 2,32).
- Environ 12 % des films (9 710 lignes) n'ont pas de genre connu.